# GIAI ĐOẠN 4: Chạy Thực Nghiệm Chính

## Experiment Design
- **2 Pipelines**: Baseline (uncertainty-only) vs Stratified (metadata + uncertainty)
- **5 Rounds**: Active Learning rounds
- **3 Seeds**: 42, 123, 456 (để có statistical significance)

## Hypotheses
- **H1**: Stratified batch có entropy cao hơn Baseline có ý nghĩa thống kê
- **H2**: Entropy cao hơn → mAP cao hơn (correlation)

## Total Runs: 6 experiments
| Run | Pipeline | Seed |
|-----|----------|------|
| 1 | Baseline | 42 |
| 2 | Baseline | 123 |
| 3 | Baseline | 456 |
| 4 | Stratified | 42 |
| 5 | Stratified | 123 |
| 6 | Stratified | 456 |

In [ ]:
# Install dependencies
!pip install ultralytics torch torchvision scikit-learn scipy pandas pyyaml -q
print("Dependencies installed!")

In [ ]:
# Clone repo
!git clone -b develop https://github.com/nguyendinhtanloc-Org/metadata-stratified-active-learning.git
import sys
sys.path.insert(0, "/kaggle/working/metadata-stratified-active-learning")

In [ ]:
# Import modules
from active_learning.loop import ActiveLearningLoop
from sampling.baseline import baseline_sampling
from sampling.stratified import stratified_sampling
import pandas as pd
import os
from datetime import datetime

In [ ]:
# Dataset paths
METADATA_PATH = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json"

IMAGE_DIRS = [
    "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainA",
    "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainB",
]

In [ ]:
# Experiment config - MAIN EXPERIMENT
EXPERIMENT_CONFIG = {
    "seed": 42,
    "initial_budget": 500,       # Initial labeled samples
    "al_batch_size": 500,        # Samples added per round
    "train_batch_size": 16,     # YOLO training batch size
    "num_rounds": 5,            # 5 AL rounds
    "epochs": 10,               # 10 epochs per round
    "device": 0,                # GPU
    "num_classes": 10,
    "stratify_fields": ["weather", "scene", "timeofday"],
    "model_name": "yolov8n"
}

print("Experiment Configuration:")
for k, v in EXPERIMENT_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Seeds cho multi-seed experiments
SEEDS = [42, 123, 456]

# Pipelines
PIPELINES = ["baseline", "stratified"]

# Results storage
all_results = []
results_dir = "results/main_experiment"
os.makedirs(results_dir, exist_ok=True)

## RUN ALL EXPERIMENTS

In [ ]:
import warnings
warnings.filterwarnings('ignore')

for pipeline in PIPELINES:
    for seed in SEEDS:
        experiment_name = f"{pipeline}_seed{seed}"
        
        print("\n" + "="*70)
        print(f"EXPERIMENT: {experiment_name}")
        print("="*70)
        
        # Create config for this run
        config = EXPERIMENT_CONFIG.copy()
        config["seed"] = seed
        
        # Monkey-patch stratified_sampling if baseline
        if pipeline == "baseline":
            import active_learning.loop as loop_module
            loop_module.stratified_sampling = baseline_sampling
            print("[INFO] Using BASELINE (uncertainty-only) sampling")
        else:
            print("[INFO] Using STRATIFIED (metadata + uncertainty) sampling")
        
        # Initialize AL loop
        al_loop = ActiveLearningLoop(
            config=config,
            experiment_name=experiment_name
        )
        
        # Run experiment
        try:
            results_df = al_loop.run(
                metadata_path=METADATA_PATH,
                image_dirs=IMAGE_DIRS
            )
            
            # Add metadata
            results_df["pipeline"] = pipeline
            results_df["seed"] = seed
            
            # Save individual results
            csv_path = f"{results_dir}/{experiment_name}.csv"
            results_df.to_csv(csv_path, index=False)
            print(f"\n[Saved] {csv_path}")
            
            # Store for combined analysis
            all_results.append(results_df)
            
            print(f"\n[DONE] {experiment_name} completed successfully!")
            
        except Exception as e:
            print(f"\n[ERROR] {experiment_name} failed: {e}")
            import traceback
            traceback.print_exc()
            continue

## SAVE COMBINED RESULTS

In [ ]:
# Combine all results
if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)
    
    # Save combined results
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    combined_path = f"{results_dir}/combined_results_{timestamp}.csv"
    combined_df.to_csv(combined_path, index=False)
    print(f"Combined results saved to: {combined_path}")
    
    # Display summary
    print("\n" + "="*70)
    print("EXPERIMENT SUMMARY")
    print("="*70)
    print(f"Total experiments: {len(all_results)}")
    print(f"\nResults preview:")
    print(combined_df.head(10))
    
    # Summary by pipeline
    print("\n" + "="*70)
    print("mAP50 by Pipeline and Round (Mean across seeds)")
    print("="*70)
    summary = combined_df.groupby(['pipeline', 'round'])['map50'].agg(['mean', 'std']).round(4)
    print(summary)
else:
    print("No results to combine!")

## VISUALIZATION

In [ ]:
import matplotlib.pyplot as plt

if all_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: mAP50 over rounds
    for pipeline in PIPELINES:
        pipeline_data = combined_df[combined_df['pipeline'] == pipeline]
        mean_by_round = pipeline_data.groupby('round')['map50'].mean()
        std_by_round = pipeline_data.groupby('round')['map50'].std()
        
        axes[0].errorbar(
            mean_by_round.index, 
            mean_by_round.values, 
            yerr=std_by_round.values,
            label=pipeline.capitalize(),
            marker='o',
            capsize=3
        )
    
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('mAP@50')
    axes[0].set_title('Learning Curve: mAP@50 over Rounds')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Number of labeled samples
    for pipeline in PIPELINES:
        pipeline_data = combined_df[combined_df['pipeline'] == pipeline]
        mean_by_round = pipeline_data.groupby('round')['labeled_count'].mean()
        
        axes[1].plot(
            mean_by_round.index, 
            mean_by_round.values, 
            label=pipeline.capitalize(),
            marker='s'
        )
    
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Labeled Samples')
    axes[1].set_title('Labeled Samples over Rounds')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{results_dir}/learning_curves.png', dpi=150)
    plt.show()
    print(f"\nPlot saved to: {results_dir}/learning_curves.png")

## STATISTICAL ANALYSIS (H1 & H2)

In [ ]:
from scipy import stats
import numpy as np

if all_results:
    print("="*70)
    print("STATISTICAL ANALYSIS")
    print("="*70)
    
    # Prepare data for paired t-test (H1)
    baseline_results = combined_df[combined_df['pipeline'] == 'baseline']
    stratified_results = combined_df[combined_df['pipeline'] == 'stratified']
    
    # Get entropy values per round (mean across seeds)
    baseline_entropy = baseline_results.groupby('round').apply(
        lambda x: x['batch_diversity'].apply(lambda d: d.get('weather', 0) if isinstance(d, dict) else 0).mean()
    ).values
    
    stratified_entropy = stratified_results.groupby('round').apply(
        lambda x: x['batch_diversity'].apply(lambda d: d.get('weather', 0) if isinstance(d, dict) else 0).mean()
    ).values
    
    print("\n[H1] Entropy Comparison (Paired t-test)")
    print("-"*40)
    if len(baseline_entropy) == len(stratified_entropy) and len(baseline_entropy) > 1:
        t_stat, p_value = stats.ttest_rel(stratified_entropy, baseline_entropy)
        print(f"  Baseline entropy mean: {np.mean(baseline_entropy):.4f}")
        print(f"  Stratified entropy mean: {np.mean(stratified_entropy):.4f}")
        print(f"  Mean difference: {np.mean(stratified_entropy - baseline_entropy):.4f}")
        print(f"  t-statistic: {t_stat:.4f}")
        print(f"  p-value: {p_value:.6f}")
        print(f"  Significant (α=0.05): {'YES' if p_value < 0.05 else 'NO'}")
    else:
        print("  Not enough data for paired t-test")
    
    # Correlation analysis (H2)
    print("\n[H2] Entropy-mAP Correlation")
    print("-"*40)
    
    # Calculate mean entropy per round
    combined_df['entropy_weather'] = combined_df['batch_diversity'].apply(
        lambda d: d.get('weather', 0) if isinstance(d, dict) else 0
    )
    
    if len(combined_df) > 2:
        pearson_r, pearson_p = stats.pearsonr(combined_df['entropy_weather'], combined_df['map50'])
        spearman_r, spearman_p = stats.spearmanr(combined_df['entropy_weather'], combined_df['map50'])
        
        print(f"  Pearson r: {pearson_r:.4f} (p={pearson_p:.6f})")
        print(f"  Spearman r: {spearman_r:.4f} (p={spearman_p:.6f})")
        print(f"  Significant (α=0.05): {'YES' if pearson_p < 0.05 else 'NO'}")
    else:
        print("  Not enough data for correlation analysis")

In [ ]:
print("\n" + "="*70)
print("MAIN EXPERIMENT COMPLETED!")
print("="*70)
print(f"\nResults saved in: {results_dir}/")
print("\nFiles:")
for f in os.listdir(results_dir):
    print(f"  - {f}")